# Simple Baseline Model — Seed Difference Logistic Regression

**Goal:** Predict P(TeamA beats TeamB) for every possible matchup in the submission file.

**Approach:** Logistic regression on seed difference — the single strongest predictor.  
Covers both men's and women's tournaments in one unified pipeline.

**Output:** `../data/submission_simple.csv` matching the SampleSubmission format.

> ⚠️ **Data required:** Run `pixi run ingest_data_into_landing` before executing this notebook.

In [13]:
import re
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

DATA_DIR  = Path("../data/landing")
MENS_DIR  = DATA_DIR / "mens"
WOMENS_DIR = DATA_DIR / "womens"
OUT_DIR   = Path("../data")

RANDOM_SEED = 42

# Verify data is present
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"
assert any(MENS_DIR.glob("*.csv")), "No men's CSVs found — run: pixi run ingest_data_into_landing"
print("✓ Data found")
print(f"  Men's files:   {len(list(MENS_DIR.glob('*.csv')))}")
print(f"  Women's files: {len(list(WOMENS_DIR.glob('*.csv')))}")

✓ Data found
  Men's files:   2
  Women's files: 2


In [14]:
# ── Load seeds and tournament results for both genders ──────────────────────

def load_gender(seeds_path: Path, results_path: Path, gender: str) -> tuple[pl.DataFrame, pl.DataFrame]:
    seeds   = pl.read_csv(seeds_path).with_columns(pl.lit(gender).alias("Gender"))
    results = pl.read_csv(results_path).with_columns(pl.lit(gender).alias("Gender"))
    return seeds, results

m_seeds, m_results = load_gender(
    MENS_DIR  / "MNCAATourneySeeds.csv",
    MENS_DIR  / "MNCAATourneyCompactResults.csv",
    gender="M",
)
w_seeds, w_results = load_gender(
    WOMENS_DIR / "WNCAATourneySeeds.csv",
    WOMENS_DIR / "WNCAATourneyCompactResults.csv",
    gender="W",
)

seeds   = pl.concat([m_seeds,   w_seeds])
results = pl.concat([m_results, w_results])

print(f"Seeds:   {seeds.shape}   Seasons: {seeds['Season'].min()}–{seeds['Season'].max()}")
print(f"Results: {results.shape}  Seasons: {results['Season'].min()}–{results['Season'].max()}")
seeds.head(3)

Seeds:   (1408, 4)   Seasons: 2015–2025
Results: (1386, 9)  Seasons: 2015–2025


Season,Seed,TeamID,Gender
i64,str,i64,str
2015,"""W01""",1124,"""M"""
2015,"""W02""",1126,"""M"""
2015,"""W03""",1108,"""M"""


In [15]:
# ── Parse seed strings → integer (e.g. 'W01' → 1, 'Z16a' → 16) ─────────────

seeds = seeds.with_columns(
    pl.col("Seed")
      .str.extract(r"(\d+)", 1)
      .cast(pl.Int32)
      .alias("SeedNum")
)

print("Seed value counts:")
print(seeds["SeedNum"].value_counts().sort("SeedNum"))

Seed value counts:
shape: (16, 2)
┌─────────┬───────┐
│ SeedNum ┆ count │
│ ---     ┆ ---   │
│ i32     ┆ u32   │
╞═════════╪═══════╡
│ 1       ┆ 88    │
│ 2       ┆ 88    │
│ 3       ┆ 88    │
│ 4       ┆ 88    │
│ 5       ┆ 88    │
│ …       ┆ …     │
│ 12      ┆ 88    │
│ 13      ┆ 88    │
│ 14      ┆ 88    │
│ 15      ┆ 88    │
│ 16      ┆ 88    │
└─────────┴───────┘


In [16]:
# ── Build symmetric matchup training rows ────────────────────────────────────
#
# Convention: Team1 = lower TeamID, Team2 = higher TeamID
# Target:     Result = 1 if Team1 (lower ID) won, else 0

def build_matchups(results: pl.DataFrame, seeds: pl.DataFrame) -> pl.DataFrame:
    df = results.with_columns([
        pl.when(pl.col("WTeamID") < pl.col("LTeamID"))
          .then(pl.col("WTeamID")).otherwise(pl.col("LTeamID")).alias("Team1"),
        pl.when(pl.col("WTeamID") < pl.col("LTeamID"))
          .then(pl.col("LTeamID")).otherwise(pl.col("WTeamID")).alias("Team2"),
        pl.when(pl.col("WTeamID") < pl.col("LTeamID"))
          .then(pl.lit(1)).otherwise(pl.lit(0)).cast(pl.Int8).alias("Result"),
    ]).select(["Season", "Gender", "Team1", "Team2", "Result"])

    seed_lookup = seeds.select(["Season", "Gender", "TeamID", "SeedNum"])

    df = (
        df
        .join(
            seed_lookup.rename({"TeamID": "Team1", "SeedNum": "T1_Seed"}),
            on=["Season", "Gender", "Team1"],
            how="left",
        )
        .join(
            seed_lookup.rename({"TeamID": "Team2", "SeedNum": "T2_Seed"}),
            on=["Season", "Gender", "Team2"],
            how="left",
        )
        .with_columns(
            (pl.col("T1_Seed") - pl.col("T2_Seed")).alias("seed_diff")
        )
        .drop_nulls(subset=["seed_diff"])
    )
    return df

train_df = build_matchups(results, seeds)

print(f"Training rows: {len(train_df)}")
print(f"  Men's:   {train_df.filter(pl.col('Gender')=='M').height}")
print(f"  Women's: {train_df.filter(pl.col('Gender')=='W').height}")
train_df.head(5)

Training rows: 1386
  Men's:   693
  Women's: 693


Season,Gender,Team1,Team2,Result,T1_Seed,T2_Seed,seed_diff
i64,str,i64,i64,i8,i32,i32,i32
2015,"""M""",1103,1129,0,1,4,-3
2015,"""M""",1111,1123,0,1,4,-3
2015,"""M""",1121,1124,0,4,1,3
2015,"""M""",1106,1137,1,4,1,3
2015,"""M""",1101,1161,1,2,3,-1


In [17]:
# ── Train logistic regression on seed_diff ───────────────────────────────────

X = train_df[["seed_diff"]].to_pandas()
y = train_df["Result"].to_pandas()

model = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="neg_log_loss")

print(f"5-fold CV log-loss: {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  (lower is better; naive 0.5 baseline = {-np.log(0.5):.4f})")

# Fit on full training data for prediction
model.fit(X, y)
print(f"\nModel coefficient (seed_diff): {model.coef_[0][0]:.4f}")
print("  (negative = lower seed number → higher win probability, as expected)")

5-fold CV log-loss: 0.6656 ± 0.0160
  (lower is better; naive 0.5 baseline = 0.6931)

Model coefficient (seed_diff): -0.2523
  (negative = lower seed number → higher win probability, as expected)


In [18]:
# ── Load sample submission — try Stage 2 first, fall back to Stage 1 ─────────

sub_path = (
    DATA_DIR / "SampleSubmissionStage2.csv"
    if (DATA_DIR / "SampleSubmissionStage2.csv").exists()
    else DATA_DIR / "SampleSubmissionStage1.csv"
)
print(f"Using submission template: {sub_path.name}")

sub_template = pl.read_csv(sub_path)
print(f"Matchups to predict: {len(sub_template)}")
print(sub_template.head(3))

Using submission template: SampleSubmissionStage1.csv
Matchups to predict: 12096
shape: (3, 2)
┌────────────────┬──────┐
│ ID             ┆ Pred │
│ ---            ┆ ---  │
│ str            ┆ f64  │
╞════════════════╪══════╡
│ 2023_1101_1102 ┆ 0.5  │
│ 2023_1101_1103 ┆ 0.5  │
│ 2023_1101_1104 ┆ 0.5  │
└────────────────┴──────┘


In [19]:
# ── Parse submission IDs and join seed features ───────────────────────────────
#
# ID format: "YYYY_TeamA_TeamB"  (TeamA < TeamB by construction)

id_parts = sub_template["ID"].str.split("_")

sub_df = sub_template.with_columns([
    id_parts.list.get(0).cast(pl.Int32).alias("Season"),
    id_parts.list.get(1).cast(pl.Int32).alias("Team1"),
    id_parts.list.get(2).cast(pl.Int32).alias("Team2"),
])

# Determine gender from the submission ID prefix logic:
# Men's TeamIDs are in one range, women's in another.
# Simplest heuristic: men's IDs < 3000, women's >= 3000 (standard Kaggle convention).
sub_df = sub_df.with_columns(
    pl.when(pl.col("Team1") < 3000).then(pl.lit("M")).otherwise(pl.lit("W")).alias("Gender")
)

seed_lookup = seeds.select(["Season", "Gender", "TeamID", "SeedNum"])

sub_df = (
    sub_df
    .join(
        seed_lookup.rename({"TeamID": "Team1", "SeedNum": "T1_Seed"}),
        on=["Season", "Gender", "Team1"],
        how="left",
    )
    .join(
        seed_lookup.rename({"TeamID": "Team2", "SeedNum": "T2_Seed"}),
        on=["Season", "Gender", "Team2"],
        how="left",
    )
    .with_columns(
        (pl.col("T1_Seed") - pl.col("T2_Seed")).fill_null(0).alias("seed_diff")
    )
)

missing_seeds = sub_df.filter(pl.col("T1_Seed").is_null() | pl.col("T2_Seed").is_null()).height
print(f"Matchups with missing seeds (filled with 0 diff → 0.5 prob): {missing_seeds}")
sub_df[["ID", "Gender", "T1_Seed", "T2_Seed", "seed_diff"]].head(5)

Matchups with missing seeds (filled with 0 diff → 0.5 prob): 0


ID,Gender,T1_Seed,T2_Seed,seed_diff
str,str,i32,i32,i32
"""2023_1101_1102""","""M""",8,5,3
"""2023_1101_1103""","""M""",8,11,-3
"""2023_1101_1104""","""M""",8,14,-6
"""2023_1101_1105""","""M""",8,12,-4
"""2023_1101_1106""","""M""",8,7,1


In [20]:
# ── Generate predictions ──────────────────────────────────────────────────────

X_sub = sub_df[["seed_diff"]].to_pandas()
preds = model.predict_proba(X_sub)[:, 1]  # P(Team1 wins)

# Clip to avoid log-loss blow-up on certain predictions
preds = np.clip(preds, 0.025, 0.975)

submission = pd.DataFrame({"ID": sub_df["ID"].to_list(), "Pred": preds})

print(f"Predictions generated: {len(submission)}")
print(submission["Pred"].describe().round(4))
print(f"\nMean prediction: {preds.mean():.4f}  (expect ~0.5 for a calibrated model)")

Predictions generated: 12096
count    12096.0000
mean         0.4799
std          0.2936
min          0.0250
25%          0.1996
50%          0.4683
75%          0.7567
max          0.9749
Name: Pred, dtype: float64

Mean prediction: 0.4799  (expect ~0.5 for a calibrated model)


In [21]:
# ── Save submission ───────────────────────────────────────────────────────────

out_path = OUT_DIR / "submission_simple.csv"
submission.to_csv(out_path, index=False)

print(f"✓ Saved: {out_path}")
print(f"  Rows: {len(submission)}")
print(f"  Columns: {list(submission.columns)}")
print()

# Verify format matches sample submission
check = pd.read_csv(out_path)
assert list(check.columns) == ["ID", "Pred"], "Column mismatch!"
assert check["Pred"].between(0.025, 0.975).all(), "Predictions out of clip range!"
assert check["ID"].equals(submission["ID"]), "ID mismatch — row order changed!"
assert len(check) == len(sub_template), f"Row count mismatch: {len(check)} vs {len(sub_template)}"

print("✓ All verification checks passed")
submission.head(10)

✓ Saved: ..\data\submission_simple.csv
  Rows: 12096
  Columns: ['ID', 'Pred']

✓ All verification checks passed


,ID,Pred
0,2023_1101_1102,0.292336
1,2023_1101_1103,0.652447
2,2023_1101_1104,0.800074
3,2023_1101_1105,0.707263
4,2023_1101_1106,0.406262
5,2023_1101_1107,0.756658
6,2023_1101_1108,0.406262
7,2023_1101_1109,0.468261
8,2023_1101_1110,0.242986
9,2023_1101_1111,0.347113
